here we made predictions and calculated metrics 

In [1]:
import os
import subprocess
import numpy as np
from PIL import Image
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
# mind the pathes before you use


# 1. Define Paths
# test_imgs_dir = "../data_goose/test/imgs"
# gt_masks_dir = "../data_goose/test/masks"
# preds_dir = "./predictions"
# checkpoint = "../checkpoints/checkpoint_epoch20.pth"

test_imgs_dir = "../data_goose/test/imgs"
gt_masks_dir = "../data_goose/test/masks"
preds_dir = "./predictions_pipeline"
checkpoint = "../checkpoints_pipeline/checkpoint_epoch13.pth"
num_classes = 12

In [3]:


os.makedirs(preds_dir, exist_ok=True)
img_files = sorted(os.listdir(test_imgs_dir))


In [ ]:
# 2. Run Batch Inference
print(f"Running inference on {len(img_files)} images (skipping already predicted ones)...")
for img_name in img_files:
    base_name = os.path.splitext(img_name)[0]
    img_path = os.path.join(test_imgs_dir, img_name)
    out_path = os.path.join(preds_dir, f"{base_name}_prediction.png")
    
    if not os.path.exists(out_path):
        # Point directly to predict.py in the parent directory
        predict_script = "predict.py"
        cmd = f"python {predict_script} --classes {num_classes} --model {checkpoint} -i {img_path} -o {out_path}"
        
        # Run the command and capture the output instead of hiding it
        result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        
        # If the command crashes, print the error and stop the loop
        if result.returncode != 0:
            print(f"CRASH DETECTED on {img_name}!")
            print(f"Error Log:\n{result.stderr}")
            break

Running inference on 962 images (skipping already predicted ones)...


In [17]:
len([f for f in os.listdir(preds_dir) if f.endswith('.png')])

962

In [ ]:
# 3. Calculate IoU
print("Calculating mIoU and per-class IoU...")
iou_per_class = {c: [] for c in range(num_classes)}

# UPDATED DIRECTORY PATH
gt_masks_dir = "../data_goose/test/masks_12class"

for img_name in img_files:
    base_name = os.path.splitext(img_name)[0]
    
    # Load Predicted Mask
    pred_path = os.path.join(preds_dir, f"{base_name}_prediction.png")
    if not os.path.exists(pred_path): continue
    pred_mask = np.array(Image.open(pred_path))
    
    # UPDATED LOADING LOGIC (PNG Only)
    gt_path = os.path.join(gt_masks_dir, f"{base_name}.png")
    gt_mask = np.array(Image.open(gt_path))
    
    # Compute Intersection over Union for each class
    for c in range(num_classes):
        pred_c = (pred_mask == c)
        gt_c = (gt_mask == c)
        
        intersection = np.logical_and(pred_c, gt_c).sum()
        union = np.logical_or(pred_c, gt_c).sum()
        
        # Only calculate if the class actually exists in the ground truth or was predicted
        if union > 0:
            iou_per_class[c].append(intersection / union)

Calculating mIoU and per-class IoU...


In [8]:
# Define the category names in the exact order of their IDs (0 to 11)
SUPER_CATEGORIES = [
    'vegetation', 'terrain', 'sky', 'construction', 
    'vehicle', 'road', 'object', 'void', 'sign', 
    'human', 'water', 'animal'
]

# 4. Aggregate Results with Names
mean_ious = {}
for c in range(num_classes):
    class_name = SUPER_CATEGORIES[c]
    if len(iou_per_class[c]) > 0:
        mean_ious[class_name] = np.mean(iou_per_class[c])
    else:
        mean_ious[class_name] = np.nan  # Class wasn't in the test set

# Calculate overall Mean IoU (ignoring NaNs)
mIoU = np.nanmean(list(mean_ious.values()))

# 5. Display as Table
df_metrics = pd.DataFrame.from_dict(mean_ious, orient='index', columns=['IoU Score'])
df_metrics.loc['Overall mIoU'] = mIoU
display(df_metrics.style.format("{:.4f}").background_gradient(cmap='viridis', subset=['IoU Score']))

,IoU Score
vegetation,0.8525
terrain,0.6912
sky,0.8175
construction,0.1672
vehicle,0.1546
road,0.0294
object,0.0029
void,0.9167
sign,0.0010
human,0.0000


In [5]:
# Define the category names in the exact order of their IDs (0 to 11)
SUPER_CATEGORIES = [
    'vegetation', 'terrain', 'sky', 'construction', 
    'vehicle', 'road', 'object', 'void', 'sign', 
    'human', 'water', 'animal'
]

# 4. Aggregate Results with Names
mean_ious = {}
for c in range(num_classes):
    class_name = SUPER_CATEGORIES[c]
    if len(iou_per_class[c]) > 0:
        mean_ious[class_name] = np.mean(iou_per_class[c])
    else:
        mean_ious[class_name] = np.nan  # Class wasn't in the test set

# Calculate overall Mean IoU (ignoring NaNs)
mIoU = np.nanmean(list(mean_ious.values()))

# 5. Display as Table
df_metrics = pd.DataFrame.from_dict(mean_ious, orient='index', columns=['IoU Score'])
df_metrics.loc['Overall mIoU'] = mIoU
display(df_metrics.style.format("{:.4f}").background_gradient(cmap='viridis', subset=['IoU Score']))

,IoU Score
vegetation,0.7995
terrain,0.5748
sky,0.7859
construction,0.0553
vehicle,0.1282
road,0.0610
object,0.0456
void,0.8657
sign,0.0482
human,0.0088


finding example for presentation

In [ ]:
# 1. Paths
test_imgs_dir = "../data_goose/test/imgs"
gt_masks_dir = "../data_goose/test/masks_12class"
preds_dir = "./predictions"

target_class = 9  # 9 is the ID for 'human'
target_img_name = None

# 2. Find an image containing the target class
print("Searching for an image containing humans...")
for file in sorted(os.listdir(gt_masks_dir)):
    if file.endswith(".png"):
        gt_mask = np.array(Image.open(os.path.join(gt_masks_dir, file)))
        if target_class in gt_mask:
            target_img_name = file
            print(f"Found! Using image: {target_img_name}")
            break

if target_img_name:
    base_name = os.path.splitext(target_img_name)[0]
    
    # Load all three images
    orig_img = np.array(Image.open(os.path.join(test_imgs_dir, target_img_name)))
    gt_mask = np.array(Image.open(os.path.join(gt_masks_dir, target_img_name)))
    pred_mask = np.array(Image.open(os.path.join(preds_dir, f"{base_name}_prediction.png")))
    
    # 3. Create Confusion Overlay
    # Find pixels where GT is 'human', but Prediction is NOT 'human'
    false_negatives = (gt_mask == target_class) & (pred_mask != target_class)
    
    # Create a copy of the original image to paint over
    confusion_overlay = orig_img.copy()
    
    # Paint the missed human pixels bright red [R, G, B]
    confusion_overlay[false_negatives] = [255, 0, 0]
    
    # Print what the model ACTUALLY guessed instead of human
    wrong_guesses = pred_mask[false_negatives]
    unique_guesses, counts = np.unique(wrong_guesses, return_counts=True)
    print("\nThe model erased the humans and replaced them with:")
    for guess, count in zip(unique_guesses, counts):
        print(f"Class ID {guess}: {count} pixels")

Searching for an image containing humans...
Found! Using image: 2022-08-30_siegertsbrunn_feldwege__0565_1661860775558704846.png

The model erased the humans and replaced them with:
Class ID 0: 63 pixels
Class ID 1: 15 pixels
Class ID 3: 7 pixels


In [ ]:
# 1. Setup Paths
test_imgs_dir = "../data_goose/test/imgs"
gt_masks_dir = "../data_goose/test/masks_12class"
preds_dir = "./predictions"

output_dir = "./presentation_assets"
os.makedirs(output_dir, exist_ok=True)

# 2. Define target image and class
target_class = 9  # Human
img_name = "2022-08-30_siegertsbrunn_feldwege__0565_1661860775558704846.png"
base_name = os.path.splitext(img_name)[0]

# 3. Load Arrays
orig_img = np.array(Image.open(os.path.join(test_imgs_dir, img_name)))
gt_mask = np.array(Image.open(os.path.join(gt_masks_dir, img_name)))
pred_mask = np.array(Image.open(os.path.join(preds_dir, f"{base_name}_prediction.png")))

# 4. Create Overlays
# 4A. Human-specific Confusion Overlay
human_false_negatives = (gt_mask == target_class) & (pred_mask != target_class)
human_confusion_overlay = orig_img.copy()
human_confusion_overlay[human_false_negatives] = [255, 0, 0]

# 4B. TOTAL Mismatch Overlay (Every single incorrect pixel)
# Find pixels where the ground truth does not match the prediction
total_mismatch = (gt_mask != pred_mask)
total_confusion_overlay = orig_img.copy()
# Paint all incorrect pixels bright red [R, G, B]
total_confusion_overlay[total_mismatch] = [255, 0, 0]

# 5. Define RGB Color Palette matching the official GOOSE dataset
palette = np.array([
    [127, 150, 143], # 0: vegetation (Muted gray-green)
    [255, 20, 126],  # 1: terrain (Hot pink)
    [220, 140, 126], # 2: sky (Salmon/peach)
    [158, 122, 186], # 3: construction (Muted purple)
    [0, 75, 73],     # 4: vehicle (Dark teal)
    [162, 0, 75],    # 5: road (Dark burgundy/magenta)
    [209, 216, 0],   # 6: object (Chartreuse/acid green)
    [255, 212, 234], # 7: void (Light pink)
    [182, 191, 84],  # 8: sign (Olive/khaki)
    [138, 168, 255], # 9: human (Periwinkle/light blue)
    [148, 162, 185], # 10: water (Gray-blue)
    [74, 62, 101],   # 11: animal (Dark navy/purple)
], dtype=np.uint8)

# Safely map the 0-11 values to RGB colors
gt_mask_colored = palette[np.clip(gt_mask, 0, 11)]
pred_mask_colored = palette[np.clip(pred_mask, 0, 11)]

# 6. Save the 5 separate images
Image.fromarray(orig_img).save(os.path.join(output_dir, "1_original.png"))
Image.fromarray(gt_mask_colored).save(os.path.join(output_dir, "2_ground_truth.png"))
Image.fromarray(pred_mask_colored).save(os.path.join(output_dir, "3_prediction.png"))
Image.fromarray(human_confusion_overlay).save(os.path.join(output_dir, "4_human_confusion_overlay.png"))
Image.fromarray(total_confusion_overlay).save(os.path.join(output_dir, "5_total_mismatch_overlay.png"))

print(f"Success! 5 presentation images have been saved to: {output_dir}")

Success! 5 presentation images have been saved to: ./presentation_assets
